# LunarAI Dataset Preparation Pipeline

**Project:** Multi-Modal Lunar Image Correspondence & Registration  
**SIH 2026 – ISRO PS26166**  
**Objective:** Build a robust dataset preparation pipeline for lunar imagery

## Sensors Covered:
- **OHRC** (Orbiter High Resolution Camera)
- **TMC** (Terrain Mapping Camera)
- **IIRS** (Imaging IR Spectrometer)
- **LRO** (Lunar Reconnaissance Orbiter)
- **KAGUYA** (SELENE)

## Pipeline Overview:
1. Scan dataset folders recursively
2. Read image files (IMG, TIFF, TIF, PNG, JPG)
3. Extract metadata (filename, dataset, sensor, dimensions, channels, file size, path)
4. Create dataset inventory table
5. Detect corrupted images
6. Detect duplicate images
7. Generate dataset statistics
8. Visualize distributions
9. Save outputs to outputs/ directory
10. Display final summary

## 1. Imports and Configuration

In [ ]:
import os
import sys
import json
import hashlib
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from typing import Dict, List, Tuple, Optional
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# Data processing
import pandas as pd
import numpy as np

# Image processing
import cv2
from PIL import Image

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

## 2. Folder Configuration

In [ ]:
# Base paths
BASE_DIR = Path(r"D:\lunar ai")
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
NOTEBOOK_DIR = BASE_DIR / "notebooks"

# Sensor directory mapping
SENSOR_MAPPING = {
    "ohrc": "OHRC",
    "tmc": "TMC",
    "iirs": "IIRS",
    "lro": "LRO",
    "kaya": "KAGUYA"
}

# Supported image extensions
IMAGE_EXTENSIONS = {'.img', '.tif', '.tiff', '.png', '.jpg', '.jpeg'}

# Create output directory if it doesn't exist
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

print(f"Base Directory: {BASE_DIR}")
print(f"Data Directory: {DATA_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")
print(f"\nSensors configured: {list(SENSOR_MAPPING.values())}")

## 3. Dataset Scanning Function

In [ ]:
def scan_dataset_directory(data_dir: Path, 
                           sensor_mapping: Dict[str, str],
                           extensions: set) -> List[Path]:
    """
    Recursively scan dataset directory for image files.
    
    Args:
        data_dir: Path to the data directory
        sensor_mapping: Dictionary mapping folder names to sensor names
        extensions: Set of supported file extensions
    
    Returns:
        List of Path objects for all found image files
    """
    image_files = []
    
    if not data_dir.exists():
        print(f"Warning: Data directory {data_dir} does not exist")
        return image_files
    
    print("Scanning dataset directory...")
    
    for sensor_folder in sensor_mapping.keys():
        sensor_path = data_dir / sensor_folder
        if not sensor_path.exists():
            print(f"  Warning: {sensor_folder} directory not found")
            continue
        
        # Recursively find all image files
        for ext in extensions:
            files = list(sensor_path.rglob(f"*{ext}"))
            image_files.extend(files)
        
        print(f"  Found {len([f for f in image_files if sensor_folder in str(f)])} files in {sensor_folder}")
    
    print(f"\nTotal image files found: {len(image_files)}")
    return image_files

## 4. Metadata Extraction Function

In [ ]:
def extract_image_metadata(image_path: Path, 
                           sensor_mapping: Dict[str, str]) -> Dict:
    """
    Extract metadata from a single image file.
    
    Args:
        image_path: Path to the image file
        sensor_mapping: Dictionary mapping folder names to sensor names
    
    Returns:
        Dictionary containing image metadata
    """
    metadata = {
        'filename': image_path.name,
        'path': str(image_path),
        'file_size': image_path.stat().st_size,
        'extension': image_path.suffix.lower(),
        'valid': True,
        'error': None
    }
    
    # Determine sensor from path
    path_str = str(image_path).lower()
    metadata['sensor'] = 'UNKNOWN'
    for folder, sensor in sensor_mapping.items():
        if folder in path_str:
            metadata['sensor'] = sensor
            break
    
    # Extract dataset from parent directory structure
    try:
        relative_path = image_path.relative_to(image_path.parents[2])
        metadata['dataset'] = str(relative_path.parts[0])
    except:
        metadata['dataset'] = 'UNKNOWN'
    
    # Try to read image dimensions and channels
    try:
        # Handle different file types
        if image_path.suffix.lower() == '.img':
            # Special handling for IMG files (likely planetary format)
            try:
                img = cv2.imread(str(image_path), cv2.IMREAD_UNCHANGED)
                if img is not None:
                    metadata['height'] = img.shape[0]
                    metadata['width'] = img.shape[1]
                    metadata['channels'] = img.shape[2] if len(img.shape) > 2 else 1
                else:
                    metadata['valid'] = False
                    metadata['error'] = 'Could not read IMG file'
                    metadata['height'] = None
                    metadata['width'] = None
                    metadata['channels'] = None
            except Exception as e:
                metadata['valid'] = False
                metadata['error'] = str(e)
                metadata['height'] = None
                metadata['width'] = None
                metadata['channels'] = None
        else:
            # Standard image formats
            with Image.open(image_path) as img:
                metadata['width'] = img.width
                metadata['height'] = img.height
                metadata['channels'] = len(img.getbands())
    except Exception as e:
        metadata['valid'] = False
        metadata['error'] = str(e)
        metadata['height'] = None
        metadata['width'] = None
        metadata['channels'] = None
    
    return metadata

## 5. Dataset Inventory Creation

In [ ]:
def create_dataset_inventory(image_files: List[Path],
                              sensor_mapping: Dict[str, str]) -> pd.DataFrame:
    """
    Create a comprehensive inventory of all images in the dataset.
    
    Args:
        image_files: List of Path objects for image files
        sensor_mapping: Dictionary mapping folder names to sensor names
    
    Returns:
        DataFrame containing metadata for all images
    """
    print("Extracting metadata from images...")
    
    inventory_data = []
    
    for image_path in tqdm(image_files, desc="Processing images"):
        try:
            metadata = extract_image_metadata(image_path, sensor_mapping)
            inventory_data.append(metadata)
        except Exception as e:
            print(f"Error processing {image_path}: {e}")
            continue
    
    # Create DataFrame
    df = pd.DataFrame(inventory_data)
    
    # Reorder columns
    columns = ['filename', 'dataset', 'sensor', 'width', 'height', 'channels', 
              'file_size', 'extension', 'path', 'valid', 'error']
    df = df.reindex(columns=columns)
    
    print(f"\nInventory created with {len(df)} records")
    return df

## 6. Corrupted Image Detection

In [ ]:
def detect_corrupted_images(inventory_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Separate corrupted and valid images from the inventory.
    
    Args:
        inventory_df: DataFrame containing image inventory
    
    Returns:
        Tuple of (corrupted_df, valid_df)
    """
    corrupted = inventory_df[~inventory_df['valid']].copy()
    valid = inventory_df[inventory_df['valid']].copy()
    
    print(f"Corrupted images: {len(corrupted)}")
    print(f"Valid images: {len(valid)}")
    
    if len(corrupted) > 0:
        print("\nCorrupted image details:")
        print(corrupted[['filename', 'sensor', 'error']].to_string(index=False))
    
    return corrupted, valid

## 7. Duplicate Image Detection

In [ ]:
def compute_file_hash(file_path: Path) -> str:
    """
    Compute MD5 hash of a file for duplicate detection.
    
    Args:
        file_path: Path to the file
    
    Returns:
        MD5 hash string
    """
    hash_md5 = hashlib.md5()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()


def detect_duplicate_images(valid_df: pd.DataFrame) -> Dict[str, List[str]]:
    """
    Detect duplicate images based on file hash.
    
    Args:
        valid_df: DataFrame containing valid images
    
    Returns:
        Dictionary mapping hash to list of file paths
    """
    print("Computing file hashes for duplicate detection...")
    
    hash_map = defaultdict(list)
    
    for _, row in tqdm(valid_df.iterrows(), total=len(valid_df), desc="Hashing files"):
        try:
            file_path = Path(row['path'])
            file_hash = compute_file_hash(file_path)
            hash_map[file_hash].append(row['path'])
        except Exception as e:
            print(f"Error hashing {row['filename']}: {e}")
            continue
    
    # Find duplicates (hashes with more than one file)
    duplicates = {h: files for h, files in hash_map.items() if len(files) > 1}
    
    print(f"\nFound {len(duplicates)} sets of duplicate files")
    total_duplicate_files = sum(len(files) for files in duplicates.values())
    print(f"Total duplicate files: {total_duplicate_files}")
    
    if duplicates:
        print("\nDuplicate sets:")
        for i, (hash_val, files) in enumerate(duplicates.items(), 1):
            print(f"  Set {i}: {len(files)} files (hash: {hash_val[:8]}...)")
            for f in files:
                print(f"    - {f}")
    
    return duplicates

## 8. Dataset Statistics Generation

In [ ]:
def generate_dataset_statistics(valid_df: pd.DataFrame) -> pd.DataFrame:
    """
    Generate comprehensive statistics for the dataset.
    
    Args:
        valid_df: DataFrame containing valid images
    
    Returns:
        DataFrame containing dataset statistics
    """
    print("Generating dataset statistics...")
    
    stats = []
    
    # Statistics by sensor
    sensor_stats = valid_df.groupby('sensor').agg({
        'filename': 'count',
        'width': ['min', 'max', 'mean'],
        'height': ['min', 'max', 'mean'],
        'file_size': ['sum', 'mean'],
        'extension': lambda x: x.mode()[0] if len(x.mode()) > 0 else 'N/A'
    }).round(2)
    
    sensor_stats.columns = ['count', 'min_width', 'max_width', 'avg_width',
                           'min_height', 'max_height', 'avg_height',
                           'total_size', 'avg_size', 'common_ext']
    
    # Overall statistics
    total_images = len(valid_df)
    total_size = valid_df['file_size'].sum()
    total_sensors = valid_df['sensor'].nunique()
    
    # File type distribution
    file_type_dist = valid_df['extension'].value_counts()
    
    # Resolution distribution (handle NaN values)
    valid_df = valid_df.copy()
    valid_df['resolution'] = valid_df['width'] * valid_df['height']
    valid_resolutions = valid_df['resolution'].dropna()
    
    if len(valid_resolutions) > 0:
        resolution_stats = {
            'min_resolution': valid_resolutions.min(),
            'max_resolution': valid_resolutions.max(),
            'avg_resolution': valid_resolutions.mean(),
            'median_resolution': valid_resolutions.median()
        }
    else:
        resolution_stats = {
            'min_resolution': 0,
            'max_resolution': 0,
            'avg_resolution': 0,
            'median_resolution': 0
        }
    
    # Create statistics DataFrame
    stats_df = pd.DataFrame({
        'metric': [
            'total_images',
            'total_sensors',
            'total_storage_bytes',
            'total_storage_mb',
            'total_storage_gb',
            'avg_file_size_bytes',
            'min_resolution',
            'max_resolution',
            'avg_resolution',
            'median_resolution'
        ],
        'value': [
            total_images,
            total_sensors,
            total_size,
            round(total_size / (1024 * 1024), 2),
            round(total_size / (1024 * 1024 * 1024), 2),
            round(valid_df['file_size'].mean(), 2),
            resolution_stats['min_resolution'],
            resolution_stats['max_resolution'],
            round(resolution_stats['avg_resolution'], 2),
            round(resolution_stats['median_resolution'], 2)
        ]
    })
    
    print("\n=== Dataset Statistics ===")
    print(stats_df.to_string(index=False))
    
    print("\n=== Statistics by Sensor ===")
    print(sensor_stats.to_string())
    
    print("\n=== File Type Distribution ===")
    print(file_type_dist.to_string())
    
    return stats_df, sensor_stats, file_type_dist

## 9. Visualization Functions

In [ ]:
def plot_dataset_distribution(valid_df: pd.DataFrame, output_path: Path):
    """
    Create a bar chart showing distribution of images by sensor.
    
    Args:
        valid_df: DataFrame containing valid images
        output_path: Path to save the plot
    """
    sensor_counts = valid_df['sensor'].value_counts().sort_values(ascending=True)
    
    plt.figure(figsize=(10, 6))
    colors = plt.cm.Set3(range(len(sensor_counts)))
    sensor_counts.plot(kind='barh', color=colors)
    plt.title('Dataset Distribution by Sensor', fontsize=14, fontweight='bold')
    plt.xlabel('Number of Images', fontsize=12)
    plt.ylabel('Sensor', fontsize=12)
    plt.grid(axis='x', alpha=0.3)
    
    # Add value labels on bars
    for i, v in enumerate(sensor_counts):
        plt.text(v + 0.5, i, str(v), va='center', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {output_path}")

In [ ]:
def plot_resolution_distribution(valid_df: pd.DataFrame, output_path: Path):
    """
    Create a histogram showing distribution of image resolutions.
    
    Args:
        valid_df: DataFrame containing valid images
        output_path: Path to save the plot
    """
    valid_df = valid_df.copy()
    valid_df['resolution'] = valid_df['width'] * valid_df['height']
    
    # Filter out NaN values
    valid_resolutions = valid_df['resolution'].dropna()
    
    if len(valid_resolutions) == 0:
        print("No valid resolution data available for plotting")
        return
    
    plt.figure(figsize=(12, 6))
    plt.hist(valid_resolutions, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    plt.title('Image Resolution Distribution', fontsize=14, fontweight='bold')
    plt.xlabel('Resolution (Width × Height)', fontsize=12)
    plt.ylabel('Frequency', fontsize=12)
    plt.grid(axis='y', alpha=0.3)
    
    # Add statistics
    mean_res = valid_resolutions.mean()
    median_res = valid_resolutions.median()
    plt.axvline(mean_res, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_res:,.0f}')
    plt.axvline(median_res, color='green', linestyle='--', linewidth=2, label=f'Median: {median_res:,.0f}')
    plt.legend()
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {output_path}")

In [ ]:
def plot_storage_usage(valid_df: pd.DataFrame, output_path: Path):
    """
    Create a pie chart showing storage usage by sensor.
    
    Args:
        valid_df: DataFrame containing valid images
        output_path: Path to save the plot
    """
    storage_by_sensor = valid_df.groupby('sensor')['file_size'].sum().sort_values(ascending=False)
    
    # Convert to MB for better readability
    storage_mb = storage_by_sensor / (1024 * 1024)
    
    plt.figure(figsize=(10, 8))
    colors = plt.cm.Set3(range(len(storage_mb)))
    wedges, texts, autotexts = plt.pie(storage_mb, labels=storage_mb.index, autopct='%1.1f%%',
                                        colors=colors, startangle=90, textprops={'fontsize': 11})
    
    # Enhance text appearance
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')
    
    plt.title('Storage Usage by Sensor (MB)', fontsize=14, fontweight='bold')
    plt.axis('equal')
    
    # Add legend with actual values
    legend_labels = [f'{sensor}: {size:.2f} MB' for sensor, size in storage_mb.items()]
    plt.legend(legend_labels, loc='center left', bbox_to_anchor=(1, 0, 0.5, 1), fontsize=10)
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {output_path}")

## 10. Save Outputs

In [ ]:
def save_outputs(inventory_df: pd.DataFrame,
                 corrupted_df: pd.DataFrame,
                 valid_df: pd.DataFrame,
                 stats_df: pd.DataFrame,
                 sensor_stats: pd.DataFrame,
                 file_type_dist: pd.Series,
                 duplicates: Dict,
                 output_dir: Path):
    """
    Save all outputs to files.
    
    Args:
        inventory_df: Complete inventory DataFrame
        corrupted_df: Corrupted images DataFrame
        valid_df: Valid images DataFrame
        stats_df: Overall statistics DataFrame
        sensor_stats: Sensor-specific statistics DataFrame
        file_type_dist: File type distribution Series
        duplicates: Dictionary of duplicate files
        output_dir: Directory to save outputs
    """
    print("\nSaving outputs...")
    
    # Save dataset inventory
    inventory_path = output_dir / "dataset_inventory.csv"
    inventory_df.to_csv(inventory_path, index=False)
    print(f"Saved: {inventory_path}")
    
    # Save dataset statistics
    stats_path = output_dir / "dataset_statistics.csv"
    stats_df.to_csv(stats_path, index=False)
    print(f"Saved: {stats_path}")
    
    # Save sensor statistics
    sensor_stats_path = output_dir / "sensor_statistics.csv"
    sensor_stats.to_csv(sensor_stats_path)
    print(f"Saved: {sensor_stats_path}")
    
    # Save file type distribution
    file_type_path = output_dir / "file_type_distribution.csv"
    file_type_dist.to_csv(file_type_path, header=['count'])
    print(f"Saved: {file_type_path}")
    
    # Save corrupted images report
    corrupted_path = output_dir / "corrupted_images.csv"
    corrupted_df.to_csv(corrupted_path, index=False)
    print(f"Saved: {corrupted_path}")
    
    # Save duplicates report
    duplicates_path = output_dir / "duplicate_images.json"
    with open(duplicates_path, 'w') as f:
        json.dump(duplicates, f, indent=2)
    print(f"Saved: {duplicates_path}")
    
    # Save comprehensive summary as JSON
    summary = {
        'timestamp': datetime.now().isoformat(),
        'total_images': len(inventory_df),
        'total_sensors': inventory_df['sensor'].nunique(),
        'total_storage_bytes': int(inventory_df['file_size'].sum()),
        'valid_images': len(valid_df),
        'corrupted_images': len(corrupted_df),
        'duplicate_sets': len(duplicates),
        'duplicate_files': sum(len(files) for files in duplicates.values()),
        'sensors': inventory_df['sensor'].unique().tolist(),
        'file_types': inventory_df['extension'].unique().tolist()
    }
    
    summary_path = output_dir / "dataset_summary.json"
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    print(f"Saved: {summary_path}")
    
    print("\nAll outputs saved successfully!")

## 11. Display Final Summary

In [ ]:
def display_final_summary(inventory_df: pd.DataFrame,
                          corrupted_df: pd.DataFrame,
                          valid_df: pd.DataFrame,
                          duplicates: Dict):
    """
    Display a comprehensive final summary of the dataset preparation.
    
    Args:
        inventory_df: Complete inventory DataFrame
        corrupted_df: Corrupted images DataFrame
        valid_df: Valid images DataFrame
        duplicates: Dictionary of duplicate files
    """
    print("\n" + "="*60)
    print("LUNARAI DATASET PREPARATION - FINAL SUMMARY")
    print("="*60)
    
    total_images = len(inventory_df)
    total_sensors = inventory_df['sensor'].nunique()
    total_storage = inventory_df['file_size'].sum()
    valid_images = len(valid_df)
    corrupted_images = len(corrupted_df)
    duplicate_sets = len(duplicates)
    duplicate_files = sum(len(files) for files in duplicates.values())
    
    print(f"\n📊 OVERALL STATISTICS")
    print(f"{'─'*40}")
    print(f"Total Images:          {total_images:,}")
    print(f"Total Sensors:         {total_sensors}")
    print(f"Total Storage:         {total_storage / (1024**3):.2f} GB ({total_storage / (1024**2):.2f} MB)")
    print(f"\n✅ VALIDITY STATUS")
    print(f"{'─'*40}")
    print(f"Valid Images:          {valid_images:,} ({valid_images/total_images*100:.1f}%)")
    print(f"Corrupted Images:      {corrupted_images:,} ({corrupted_images/total_images*100:.1f}%)")
    print(f"\n🔍 DUPLICATE DETECTION")
    print(f"{'─'*40}")
    print(f"Duplicate Sets:         {duplicate_sets}")
    print(f"Duplicate Files:        {duplicate_files:,}")
    
    print(f"\n📡 SENSOR BREAKDOWN")
    print(f"{'─'*40}")
    sensor_counts = inventory_df['sensor'].value_counts()
    for sensor, count in sensor_counts.items():
        percentage = count / total_images * 100
        storage = inventory_df[inventory_df['sensor'] == sensor]['file_size'].sum()
        print(f"{sensor:12s}: {count:6,} images ({percentage:5.1f}%) - {storage/(1024**2):8.2f} MB")
    
    print(f"\n📁 FILE TYPE DISTRIBUTION")
    print(f"{'─'*40}")
    file_type_counts = inventory_df['extension'].value_counts()
    for ext, count in file_type_counts.items():
        percentage = count / total_images * 100
        print(f"{ext:8s}: {count:6,} files ({percentage:5.1f}%)")
    
    print(f"\n" + "="*60)
    print(f"Pipeline completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*60)

## 12. Main Pipeline Execution

In [ ]:
def main():
    """
    Main execution function for the dataset preparation pipeline.
    """
    print("="*60)
    print("LUNARAI DATASET PREPARATION PIPELINE")
    print("="*60)
    print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print()
    
    try:
        # Step 1: Scan dataset directory
        print("\n" + "="*60)
        print("STEP 1: Scanning Dataset Directory")
        print("="*60)
        image_files = scan_dataset_directory(DATA_DIR, SENSOR_MAPPING, IMAGE_EXTENSIONS)
        
        if not image_files:
            print("ERROR: No image files found. Please check the data directory.")
            return
        
        # Step 2: Create dataset inventory
        print("\n" + "="*60)
        print("STEP 2: Creating Dataset Inventory")
        print("="*60)
        inventory_df = create_dataset_inventory(image_files, SENSOR_MAPPING)
        
        # Step 3: Detect corrupted images
        print("\n" + "="*60)
        print("STEP 3: Detecting Corrupted Images")
        print("="*60)
        corrupted_df, valid_df = detect_corrupted_images(inventory_df)
        
        # Step 4: Detect duplicate images
        print("\n" + "="*60)
        print("STEP 4: Detecting Duplicate Images")
        print("="*60)
        duplicates = detect_duplicate_images(valid_df)
        
        # Step 5: Generate statistics
        print("\n" + "="*60)
        print("STEP 5: Generating Dataset Statistics")
        print("="*60)
        stats_df, sensor_stats, file_type_dist = generate_dataset_statistics(valid_df)
        
        # Step 6: Create visualizations
        print("\n" + "="*60)
        print("STEP 6: Creating Visualizations")
        print("="*60)
        plot_dataset_distribution(valid_df, OUTPUT_DIR / "dataset_distribution.png")
        plot_resolution_distribution(valid_df, OUTPUT_DIR / "resolution_distribution.png")
        plot_storage_usage(valid_df, OUTPUT_DIR / "storage_usage.png")
        
        # Step 7: Save outputs
        print("\n" + "="*60)
        print("STEP 7: Saving Outputs")
        print("="*60)
        save_outputs(inventory_df, corrupted_df, valid_df, stats_df, 
                   sensor_stats, file_type_dist, duplicates, OUTPUT_DIR)
        
        # Step 8: Display final summary
        print("\n" + "="*60)
        print("STEP 8: Final Summary")
        print("="*60)
        display_final_summary(inventory_df, corrupted_df, valid_df, duplicates)
        
        print("\n" + "="*60)
        print("✅ PIPELINE COMPLETED SUCCESSFULLY")
        print("="*60)
        
    except Exception as e:
        print(f"\n❌ ERROR: Pipeline failed with exception:")
        print(f"{type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
        
    print(f"\nFinished at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 13. Execute Pipeline

In [ ]:
# Run the main pipeline
if __name__ == "__main__":
    main()

---

## Pipeline Complete!

This notebook has successfully:

1. ✅ Scanned all dataset folders recursively
2. ✅ Read image files from multiple formats (IMG, TIFF, TIF, PNG, JPG)
3. ✅ Extracted comprehensive metadata for each image
4. ✅ Created a complete dataset inventory table
5. ✅ Detected and reported corrupted images
6. ✅ Detected duplicate images using MD5 hashing
7. ✅ Generated detailed dataset statistics
8. ✅ Created visualization charts for dataset insights
9. ✅ Saved all outputs to the outputs/ directory
10. ✅ Displayed a comprehensive final summary

### Output Files Generated:
- `dataset_inventory.csv` - Complete inventory of all images
- `dataset_statistics.csv` - Overall dataset statistics
- `sensor_statistics.csv` - Statistics broken down by sensor
- `file_type_distribution.csv` - Distribution of file types
- `corrupted_images.csv` - List of corrupted images
- `duplicate_images.json` - Detailed duplicate file information
- `dataset_summary.json` - Comprehensive summary in JSON format
- `dataset_distribution.png` - Bar chart of images by sensor
- `resolution_distribution.png` - Histogram of image resolutions
- `storage_usage.png` - Pie chart of storage usage by sensor

### Next Steps:
- Review the generated outputs in the `outputs/` directory
- Analyze the statistics and visualizations for insights
- Address any corrupted images if found
- Consider removing or organizing duplicate files
- Use this inventory for subsequent data processing pipelines